# `klygo.config` (`structure.py`): Flatten and unflatten nested configurations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/config/Document_Config_Structure.ipynb)


Convert nested dictionaries into flat path/value mappings and reconstruct nested dictionaries while detecting path collisions.

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare workspace

In [1]:
from tempfile import TemporaryDirectory
from klygo import config, files

# Run this cell again to replace only this notebook's temporary workspace.
if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-config-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-config-05rjm23l


## Jump to an API

[`flatten`](#flatten) · [`unflatten`](#unflatten)

<a id="flatten"></a>

### `config.flatten`

Flatten nested mappings into leaf paths.

**Signature:** `config.flatten(config_data: Mapping[str, Any] | box.Box, sep: str = '.') -> dict[str, Any]`

In [2]:
nested = {"model": {"batch": 16, "extra": {}}, "seed": 7}
flat_dot = config.flatten(nested)
flat_dunder = config.flatten(nested, sep="__")
print("Dot flattened:", flat_dot)
print("Custom separator:", flat_dunder)

Dot flattened: {'model.batch': 16, 'model.extra': {}, 'seed': 7}
Custom separator: {'model__batch': 16, 'model__extra': {}, 'seed': 7}


<a id="unflatten"></a>

### `config.unflatten`

Expand flat key paths into nested dictionaries.

**Signature:** `config.unflatten(flat_dict: Mapping[str, Any], sep: str = '.') -> dict[str, Any]`

In [3]:
reconstructed = config.unflatten(flat_dunder, sep="__")
print("Reconstructed matches original:", reconstructed == nested)
try:
    config.unflatten({"model": 1, "model.batch": 16})
except ValueError as exc:
    print("ValueError on conflicting paths ->", exc)

Reconstructed matches original: True
ValueError on conflicting paths -> conflicting configuration path: 'model.batch'


## Put it together

In [4]:
nested = {"train": {"optimizer": {"name": "adamw", "lr": 0.001}}}
flat_map = config.flatten(nested)
flat_map["train.optimizer.weight_decay"] = 0.01
restored = config.unflatten(flat_map)
print("Restored optimizer:", restored["train"]["optimizer"])

Restored optimizer: {'name': 'adamw', 'lr': 0.001, 'weight_decay': 0.01}


## Remember

- `flatten()` and `unflatten()` convert between nested mappings and single-level dot-path dictionaries, raising `ValueError` on separator or leaf-vs-mapping conflicts.
- Next in `klygo.config`: continue with [Document_Config_Environment.ipynb](Document_Config_Environment.ipynb) or return to [Document_Config_Overview.ipynb](Document_Config_Overview.ipynb).
- Related Klygo modules: [`klygo.files`](../files/Document_Files_Overview.ipynb) · [`klygo.archive`](../archive/Document_Archive_Overview.ipynb).